<a href="https://colab.research.google.com/github/rudalshan0412-code/Intent_Classifier-RAG_Chatbot/blob/main/01)_%EB%AC%B8%EC%84%9C_%EA%B2%80%EC%83%89_%ED%94%84%EB%A1%9C%ED%86%A0%ED%83%80%EC%9E%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
문서 불러오기
→ 문서 정리
→ 작은 단위로 분할
→ 각 조각을 벡터로 변환
→ 질문을 벡터로 변환
→ 가장 비슷한 문서 조각 검색
→ 검색 결과 출력
'''

'\n문서 불러오기\n→ 문서 정리\n→ 작은 단위로 분할\n→ 각 조각을 벡터로 변환\n→ 질문을 벡터로 변환\n→ 가장 비슷한 문서 조각 검색\n→ 검색 결과 출력\n'

In [ ]:
from pathlib import Path

# 텍스트 데이터 불러오기 전 검증

def load_text_file(file_path: str) -> str:
# -> 는 반환할 값의 타입을 알려준다(강제성은 없다)
    path = Path(file_path)
    # Path()는 경로 데이터가 저장된다(경로와 관련된 다른 기능도 사용 가능하다)

    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}")

    if path.suffix.lower() != ".txt": # path.suffix는 파일의 확장자를 추출한다
        raise ValueError("현재는 TXT 파일만 지원합니다.")

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="cp949")

# 파일이 존재하는지, txt 파일인지 검증하여 해당하지 않을 경우 Error을 발생시킨다.

In [ ]:
# 정규표현식 전처리

In [ ]:
import re


def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    # 여러 개의 공백, 줄바꿈, 탭을 하나의 공백으로 바꾼다

    text = re.sub(r"[^\w\s가-힣.,!?()\-]", "", text)
    # 한글, 영문, 숫자, 공백, 일부 문장부호를 제외한 문자를 제거한다.
    # \w 는 영문, 숫자, 밑줄 \s 는 공백문자, -는 범위, \-는 하이픈을 의미한다.

    return text.strip()

In [ ]:
# 문서 분할

In [ ]:
def split_text(text: str, chunk_size: int = 300, overlap: int = 50,) -> list[str]:
# 한번에 자를 크기는 300(chunk_size), 50씩 겹치게 자른다(overlap)
    if chunk_size <= 0:
        raise ValueError("chunk_size는 1 이상이어야 합니다.")

    if overlap < 0:
        raise ValueError("overlap은 0 이상이어야 합니다.")

    if overlap >= chunk_size:
        raise ValueError("overlap은 chunk_size보다 작아야 합니다.")

    chunks: list[str] = [] # 쪼개진 조각들을 담을 리스트
    start = 0 # 자를 첫번째 글자의 인덱스

    while start < len(text):
        end = start + chunk_size # 끝 지점을 정해준다(시작점부터 자르는 크기)
        chunk = text[start:end].strip() # 슬라이싱 후 앞 뒤 공백 제거

        if chunk: # 잘라낸 조각이 비어있지 않으면
            chunks.append(chunk) # 결과 리스트에 추가한다

        start += chunk_size - overlap # 다음 시작 지점을 overlap만큼 빼고 다시 설정

    return chunks

In [ ]:
# TF-IDF 검사기
# TF-IDF는 각 단어가 문서 내에서 얼마나 중요한지를 나타낸다

In [ ]:
from dataclasses import dataclass
# 데이터 저장용 클래스 관련 패키지
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


@dataclass
# __init__, __repr__ 를 생략하게 해줌
# 예를 들어, Searchresult("aaa", 90.0, 1) 이면 이를 순서대로 처리해줌
class SearchResult:
    text: str
    score: float
    index: int


class TfidfRetriever:

    # 텍스트 데이터 백터화
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer()
        # TfidfVectorizer.fit_transform() 하면 분석 및 변환
        self.chunks: list[str] = []
        self.chunk_vectors = None

    # 학습 단계

    def fit(self, chunks: list[str]) -> None:
        if not chunks:
            raise ValueError("검색할 문서 청크가 없습니다.")

        self.chunks = chunks
        self.chunk_vectors = self.vectorizer.fit_transform(chunks)
        # 여기서 문서 분석 및 벡터 표 생성

  # 검색 단계

    # 예외 처리 및 검사
    def search(
        self,
        query: str,
        top_k: int = 3,
    ) -> list[SearchResult]:
    # top_k 기본값은 3
        if self.chunk_vectors is None:
            raise RuntimeError("먼저 fit()을 실행해야 합니다.")

        if not query.strip(): # 만약 공백인 경우
            raise ValueError("질문을 입력해야 합니다.")

        # 질문 벡터 변환 및 유사도 검사

        query_vector = self.vectorizer.transform([query])
        # TfidfVectorizer.transform은 변환만 시킨다(분석까지는 X)
        # .fit()이 분석만 한다
        similarities = cosine_similarity(
            query_vector,
            self.chunk_vectors,
        )[0]
        # cosine_similarity()는 0(전혀 유사하지 않음) ~ 1(매우 유사함)으로 나타남


        # 순위 정렬 및 결과 가공
        sorted_indices = similarities.argsort()[::-1][:top_k]
        # 유사도 점수를 높은 순서대로 인덱스 정렬
        # 상위 top_k 개 만큼만 잘라냄
        results = []

        for index in sorted_indices:
            results.append(
                SearchResult(
                    text=self.chunks[index],
                    score=float(similarities[index]),
                    index=int(index),
                )
            )
            # 처음에 선언한 SearchResult()의 구조에 담아서 반환한다

        return results

In [ ]:
# 실행 코드

In [ ]:
from src.chunker import split_text
from src.document_loader import load_text_file
from src.retriever import TfidfRetriever
from src.text_preprocessor import clean_text


def main() -> None:
    document = load_text_file("data/sample.txt")
    cleaned_document = clean_text(document)

    chunks = split_text(
        cleaned_document,
        chunk_size=300,
        overlap=50,
    )

    retriever = TfidfRetriever()
    retriever.fit(chunks)

    print(f"문서 청크 개수: {len(chunks)}")

    while True:
        query = input("\n질문을 입력하세요. 종료하려면 exit 입력: ").strip()

        if query.lower() == "exit":
            print("프로그램을 종료합니다.")
            break

        try:
            results = retriever.search(query, top_k=3)
        except ValueError as error:
            print(error)
            continue

        for rank, result in enumerate(results, start=1):
            print(f"\n[{rank}위]")
            print(f"유사도: {result.score:.4f}")
            print(result.text)


if __name__ == "__main__":
    main()

ImportError: cannot import name 'split_text' from 'src.rag.chunker' (/content/drive/MyDrive/rag_intent_chatbot/src/rag/chunker.py)